# RoadSense: Improve the model (v2)

Version 1 (YOLO26**n**, RDD2022 India + Japan) reached **mAP50 0.528**. It had two weaknesses we saw on the live site:
1. **Close-up photos.** All training photos were taken from a car (dashcam view), so a phone photo
   taken standing right over a pothole was often missed.
2. **False alarms** on things that look like cracks, like brick footpaths.

This notebook makes **v2** and, most importantly, **measures** whether it is really better:

| Change | Why |
|---|---|
| Bigger model: YOLO26**s** instead of **n** | about 3× more capacity; still fast enough for our server (tested: ~0.2 s per photo on a normal CPU) |
| Add **665 close-up / mixed-angle pothole photos** | teaches the model what potholes look like from a phone |
| Test on **three** things, old vs new | the normal validation set, close-up photos it never saw, and clean roads (false alarms) |

**Dataset credit:** Pothole dataset by Atikur Rahman Chitholian (665 images), licence **ODbL v1.0**,
via Roboflow Universe (`brad-dwyer/pothole-voxrl`). You need a **free Roboflow account** for the download key (Step 4).

### Before you start
- Menu **Runtime → Change runtime type → T4 GPU → Save**.
- Run the cells **one at a time, top to bottom** (Shift + Enter).
- Training (Step 8) takes **about 4–6 hours**. It saves after every epoch to Google Drive, so a disconnect loses nothing (Step 8b).

## Step 1: GPU and Google Drive

In [ ]:
# You should see "Tesla T4". If you see an error: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/roadsense'      # the same folder as Day 1
OLD_BEST = f'{DRIVE_DIR}/runs/rdd_y26n/weights/best.pt'
print('Old model found:', os.path.exists(OLD_BEST), '->', OLD_BEST)
print('RDD copy found :', os.path.exists(f'{DRIVE_DIR}/rdd_india_japan.zip'))

## Step 2: Install the libraries

In [ ]:
# roboflow = the small library that downloads the pothole dataset
!pip install -q -U ultralytics onnx onnxruntime roboflow
import ultralytics
ultralytics.checks()

## Step 3: Restore RDD2022 (India + Japan) from your Drive and convert it

This is the **same data and the same 85/15 split as Day 1** (same code, same random seed),
so the old and new models are tested on exactly the same validation photos. About 5 minutes.

In [ ]:
import os, glob, random, shutil, zipfile
import xml.etree.ElementTree as ET
from PIL import Image

RAW = '/content/raw'
CACHE = f'{DRIVE_DIR}/rdd_india_japan.zip'
if not (os.path.isdir(f'{RAW}/India') and os.path.isdir(f'{RAW}/Japan')):
    print('Unzipping India + Japan from Drive ...')
    !unzip -q -o "{CACHE}" -d {RAW}

CLASS_MAP = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}
CLASS_NAMES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
POTHOLE = 3

def find_image_for_xml(xml_path):
    stem = os.path.splitext(os.path.basename(xml_path))[0]
    train_dir = xml_path.split('/annotations/')[0]
    for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
        p = os.path.join(train_dir, 'images', stem + ext)
        if os.path.exists(p):
            return p
    return None

def voc_to_yolo_lines(xml_path, img_path):
    root = ET.parse(xml_path).getroot()
    try:
        W = int(float(root.findtext('size/width'))); H = int(float(root.findtext('size/height')))
    except (TypeError, ValueError):
        W = H = 0
    if W <= 0 or H <= 0:
        with Image.open(img_path) as im:
            W, H = im.size
    lines = []
    for obj in root.iter('object'):
        name = obj.findtext('name', '').strip()
        if name not in CLASS_MAP:
            continue
        bb = obj.find('bndbox')
        x1, y1 = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
        x2, y2 = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
        x1, x2 = max(0, min(x1, W)), max(0, min(x2, W))
        y1, y2 = max(0, min(y1, H)), max(0, min(y2, H))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        lines.append(f'{CLASS_MAP[name]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
    return lines

def build_yolo_dataset(raw_dir, out_dir, val_fraction=0.15, seed=42):
    random.seed(seed)
    for split in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{split}', exist_ok=True)
    xmls = sorted(glob.glob(f'{raw_dir}/**/train/**/*.xml', recursive=True))
    by_country = {}
    for x in xmls:
        by_country.setdefault(os.path.relpath(x, raw_dir).split('/')[0], []).append(x)
    stats = {'train': 0, 'val': 0, 'missing_image': 0, 'boxes': 0}
    for country, files in by_country.items():
        random.shuffle(files)
        n_val = int(len(files) * val_fraction)
        for i, xml_path in enumerate(files):
            split = 'val' if i < n_val else 'train'
            img = find_image_for_xml(xml_path)
            if img is None:
                stats['missing_image'] += 1
                continue
            lines = voc_to_yolo_lines(xml_path, img)
            stem = f"{country}_{os.path.splitext(os.path.basename(img))[0]}"
            shutil.copy(img, f'{out_dir}/images/{split}/{stem}{os.path.splitext(img)[1].lower()}')
            with open(f'{out_dir}/labels/{split}/{stem}.txt', 'w') as f:
                f.write('\n'.join(lines))
            stats[split] += 1
            stats['boxes'] += len(lines)
    return stats

DATA = '/content/rs_v2'                      # the combined dataset for v2
if os.path.exists(DATA):
    shutil.rmtree(DATA)
stats = build_yolo_dataset(RAW, DATA)
print(stats)        # Day 1 had train 15,482 and val 2,730: these must match

## Step 4: Download the close-up pothole dataset

1. Go to **roboflow.com** → **Sign up** (free; "Continue with Google" is fine).
2. Open **https://app.roboflow.com/settings/api** (or: your workspace → Settings → API Keys) and **copy the Private API Key**.
3. Run the cell below and **paste the key into the box** that appears, then press Enter.

The key is typed into a hidden box, so it isn't saved in the notebook. **Never paste it into a code line or share it.**

In [ ]:
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('Paste your Roboflow Private API Key and press Enter: '))
PH = '/content/pothole_rf'
ds = rf.workspace('brad-dwyer').project('pothole-voxrl').version(1).download('yolov8', location=PH)

import yaml, glob
info = yaml.safe_load(open(f'{PH}/data.yaml'))
print('classes in this dataset:', info.get('names'))
for split in ('train', 'valid', 'test'):
    print(split, len(glob.glob(f'{PH}/{split}/images/*')), 'images')

## Step 5: Add the pothole photos to our dataset

- This dataset has one class (pothole). In our model, pothole is class **3**, so every label is renamed to 3.
- Its **train** photos (465) go into our training set. Our training set has 15,482 photos, so 465 would be a
  drop in the ocean: we add each one **3 times** (the random flips, crops and colour changes during training
  make every copy look different). This is called **oversampling**.
- Its **valid + test** photos (200) are **never trained on**. They become a separate exam, `closeup_val`,
  that measures exactly the weakness we want to fix.

In [ ]:
import os, glob, shutil

def copy_relabel(src_split, dst_img, dst_lbl, repeats=1, prefix='ph'):
    os.makedirs(dst_img, exist_ok=True); os.makedirs(dst_lbl, exist_ok=True)
    n_img = n_box = 0
    for img in sorted(glob.glob(f'{PH}/{src_split}/images/*')):
        stem, ext = os.path.splitext(os.path.basename(img))
        lbl = f'{PH}/{src_split}/labels/{stem}.txt'
        lines = []
        if os.path.exists(lbl):
            for line in open(lbl):
                parts = line.split()
                if len(parts) == 5:                       # class x y w h  ->  3 x y w h
                    lines.append(' '.join([str(POTHOLE)] + parts[1:]))
        for r in range(repeats):
            name = f'{prefix}_{stem}_r{r}'
            shutil.copy(img, f'{dst_img}/{name}{ext.lower()}')
            with open(f'{dst_lbl}/{name}.txt', 'w') as f:
                f.write('\n'.join(lines))
            n_img += 1; n_box += len(lines)
    return n_img, n_box

assert len(info['names']) == 1, 'Expected a single pothole class: send Claude the class list printed above.'
print('added to training:', copy_relabel('train', f'{DATA}/images/train', f'{DATA}/labels/train', repeats=3))
CLOSE = '/content/closeup_val'
if os.path.exists(CLOSE):
    shutil.rmtree(CLOSE)
for split in ('valid', 'test'):
    print('close-up exam  :', copy_relabel(split, f'{CLOSE}/images/val', f'{CLOSE}/labels/val', repeats=1, prefix=split))

import yaml
names = {i: n for i, n in enumerate(CLASS_NAMES)}
yaml.safe_dump({'path': DATA, 'train': 'images/train', 'val': 'images/val', 'names': names},
               open(f'{DATA}/data.yaml', 'w'), sort_keys=False)
yaml.safe_dump({'path': CLOSE, 'train': 'images/val', 'val': 'images/val', 'names': names},
               open(f'{CLOSE}/data.yaml', 'w'), sort_keys=False)
print('training photos now:', len(glob.glob(f'{DATA}/images/train/*')))

## Step 6: Look at some of the new photos with their boxes

In [ ]:
import random, glob, os
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
imgs = sorted(glob.glob(f'{DATA}/images/train/ph_*_r0.*'))
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, p in zip(axes.flat, random.sample(imgs, 6)):
    im = Image.open(p).convert('RGB'); W, H = im.size; d = ImageDraw.Draw(im)
    for line in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
        c, xc, yc, w, h = map(float, line.split())
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='red', width=4)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()

## Step 7: Measure the OLD model first (the baseline)

Three exams. We run the exact same three on the new model later.
1. **RDD validation** (2,730 dashcam photos): the score from Day 1, so it should be about **0.528**.
2. **Close-up exam** (200 pothole photos never used in training): pothole AP50.
3. **False-alarm test**: the **1,029** validation photos that contain **no damage at all**. What share of them
   get at least one box at the website's threshold (30% confidence)? Lower is better.

In [ ]:
import json, glob, os
from ultralytics import YOLO

APP_CONF = 0.30          # the same confidence threshold the website uses (CONF_THRESHOLD in config.py)

def background_images():
    out = []
    for img in sorted(glob.glob(f'{DATA}/images/val/*')):
        lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lbl) and os.path.getsize(lbl) == 0:
            out.append(img)
    return out

def false_alarm_rate(model, imgs, conf=APP_CONF):
    flagged = 0
    for i in range(0, len(imgs), 32):
        for r in model.predict(imgs[i:i+32], imgsz=640, conf=conf, verbose=False):
            flagged += int(len(r.boxes) > 0)
    return flagged / len(imgs)

def evaluate(weights, label):
    m = YOLO(weights)
    rdd = m.val(data=f'{DATA}/data.yaml', imgsz=640, split='val', verbose=False, plots=False)
    close = m.val(data=f'{CLOSE}/data.yaml', imgsz=640, split='val', verbose=False, plots=False)
    per_class = {CLASS_NAMES[int(c)]: round(float(rdd.box.ap50[i]), 4) for i, c in enumerate(rdd.box.ap_class_index)}
    close_pothole = [float(close.box.ap50[i]) for i, c in enumerate(close.box.ap_class_index) if int(c) == POTHOLE][0]
    bg = background_images()
    res = {'model': label,
           'rdd_mAP50': round(float(rdd.box.map50), 4), 'rdd_mAP50_95': round(float(rdd.box.map), 4),
           'rdd_precision': round(float(rdd.box.mp), 4), 'rdd_recall': round(float(rdd.box.mr), 4),
           'rdd_AP50_per_class': per_class,
           'closeup_pothole_AP50': round(close_pothole, 4),
           'background_images': len(bg),
           'false_alarm_rate': round(false_alarm_rate(m, bg), 4)}
    print(json.dumps(res, indent=2))
    return res

old = evaluate(OLD_BEST, 'v1: YOLO26n, RDD only')
json.dump(old, open(f'{DRIVE_DIR}/metrics_v1_baseline.json', 'w'), indent=2)

## Step 8: Train v2 (about 4–6 hours)

- `yolo26s.pt`: the **small** model, starting from weights pre-trained on everyday photos (COCO).
- `epochs=40, patience=10`: at most 40 rounds; stop early if validation hasn't improved for 10.
- Everything is saved to **Drive → roadsense/runs/rdd_y26s_v2** after every epoch.

Leave the tab open. If it disconnects, use **Step 8b**.

In [ ]:
from ultralytics import YOLO
RUN_DIR = f'{DRIVE_DIR}/runs'
model = YOLO('yolo26s.pt')
model.train(data=f'{DATA}/data.yaml', epochs=40, patience=10, imgsz=640, batch=16,
            project=RUN_DIR, name='rdd_y26s_v2', exist_ok=True, seed=42)

### Step 8b: ONLY if Colab disconnected during training
1. Runtime → Change runtime type → T4 GPU (if needed).
2. Run **Steps 1, 2, 3, 4 and 5** again (the dataset lived on the temporary disk). Skip 6 and 7:
   the baseline is already saved in Drive.
3. Run the cell below. It continues from the last saved epoch.

In [ ]:
from ultralytics import YOLO
YOLO(f'{DRIVE_DIR}/runs/rdd_y26s_v2/weights/last.pt').train(resume=True)

## Step 9: Measure the NEW model on the same three exams, and compare

In [ ]:
import json, os
NEW_BEST = f'{DRIVE_DIR}/runs/rdd_y26s_v2/weights/best.pt'
old = json.load(open(f'{DRIVE_DIR}/metrics_v1_baseline.json'))
new = evaluate(NEW_BEST, 'v2: YOLO26s, RDD + close-up potholes')
json.dump(new, open(f'{DRIVE_DIR}/metrics_v2.json', 'w'), indent=2)

rows = [('RDD validation mAP50 (higher = better)', 'rdd_mAP50'),
        ('RDD validation mAP50-95', 'rdd_mAP50_95'),
        ('Close-up pothole AP50 (higher = better)', 'closeup_pothole_AP50'),
        ('False alarms on clean roads (lower = better)', 'false_alarm_rate')]
print(f"{'':48}{'v1 (old)':>10}{'v2 (new)':>10}")
for label, k in rows:
    print(f'{label:48}{old[k]:>10.3f}{new[k]:>10.3f}')
print('\nPer damage type (RDD AP50):')
for c in CLASS_NAMES:
    print(f"  {c:22}{old['rdd_AP50_per_class'].get(c, 0):>8.3f}{new['rdd_AP50_per_class'].get(c, 0):>8.3f}")

better = (new['rdd_mAP50'] >= old['rdd_mAP50'] - 0.01 and
          new['closeup_pothole_AP50'] > old['closeup_pothole_AP50'] and
          new['false_alarm_rate'] <= old['false_alarm_rate'] + 0.02)
print('\nVERDICT:', 'v2 is better: use it' if better else 'v2 is NOT clearly better: keep v1 and send Claude this table')

## Step 10: See v2 on the close-up photos it never saw

In [ ]:
import glob, random
import matplotlib.pyplot as plt
from ultralytics import YOLO
best = YOLO(NEW_BEST)
imgs = random.sample(sorted(glob.glob(f'{CLOSE}/images/val/*')), 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, r in zip(axes.flat, best.predict(imgs, imgsz=640, conf=APP_CONF, verbose=False)):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(f'{len(r.boxes)} detections'); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/sample_predictions_v2.png', dpi=80); plt.show()

## Step 11: Export v2 to ONNX and check it

The website runs the model with `onnxruntime` on a CPU. This cell exports v2, checks the output shape
(our server code needs it) and times both models on Colab's CPU.

In [ ]:
import shutil, os, time, numpy as np, onnxruntime as ort
onnx_path = YOLO(NEW_BEST).export(format='onnx', imgsz=640)
final = f'{DRIVE_DIR}/roadsense_v2.onnx'
shutil.copy(onnx_path, final)

def cpu_ms(path):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s = ort.InferenceSession(path, so, providers=['CPUExecutionProvider'])
    x = np.random.rand(1, 3, 640, 640).astype(np.float32); name = s.get_inputs()[0].name
    out = s.run(None, {name: x}); t = time.time()
    for _ in range(5):
        s.run(None, {name: x})
    return (time.time() - t) / 5 * 1000, out[0].shape

for label, p in [('v1', f'{DRIVE_DIR}/roadsense.onnx'), ('v2', final)]:
    if os.path.exists(p):
        ms, shape = cpu_ms(p)
        print(f'{label}: output shape {shape}, {ms:.0f} ms per photo on 1 CPU thread, {os.path.getsize(p)/1e6:.1f} MB')
print('\nSaved:', final)

## Done: what to send Claude

1. The **comparison table and VERDICT** from Step 9.
2. The **two output-shape lines** from Step 11.

Your Drive folder **roadsense/** now also has `roadsense_v2.onnx`, `metrics_v1_baseline.json`, `metrics_v2.json`
and `sample_predictions_v2.png`. If the verdict says v2 is better, the next step is copying `roadsense_v2.onnx`
into the backend (Claude will give you the exact steps).